# G04: Scaling

*Does the grounding gain grow, shrink or stay flat as a model family scales from 1B to 405B parameters?*

| Sub-metric | Definition | Quantification | Scoring |
|---|---|---|---|
| **Capacity series** | MRR per arm at each rung of the Llama ladder, on the `main` family | paired bootstrap over queries, as in G03 | reported per rung |
| **Pooled versus grounded lift** | the KG-arm lift over `none` on every pair, and on the pairs the graph has evidence for | one row per rung with both lifts and their intervals | the grounded lift is the capability reading |

A larger model has memorised more approvals, which raises the `none` arm and mechanically shrinks the pooled lift. A graph with no evidence for a pair cannot help however capable the model, so the grounded-arm lift is what speaks to the model's ability to use evidence.

**Reported, not scored.** The per-graph breakdown by rung.

**Inputs:** `results/tables/G02_run/runs/` (the capacity models) · `results/checkpoints/G01_benchmark.pkl` · `config/params.yaml` (`grounding.capacity_models`)

**Outputs:** `results/{figures,tables}/G04_scaling/` · `results/checkpoints/G04_scaling.pkl`

Design notes and justifications: [`README.md`](README.md)

## Set-up

Loads only the rungs that have been run; the tables fill in as more rungs finish.

In [1]:
# Pick up edits to src/ without restarting the kernel. Without this, editing a
# module after it has been imported leaves the kernel holding the old version,
# which surfaces as a confusing ImportError for a name that is plainly on disk.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass                      # not running under IPython (headless runner, plain python)

import gc
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Locate the repo root from any depth, then make the package importable.
# (`pip install -e .` also works and makes this block a no-op.)
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'config' / 'kgs.yaml').exists())
if str(_root / 'src') not in sys.path:
    sys.path.insert(0, str(_root / 'src'))

from IPython.display import display
import numpy as np
import pandas as pd


from biokgsuite.grounding import capacity_table, chance_mrr, load_runs, scaling_table, validate_runs
from biokgsuite.viz.grounding import fig_capacity, fig_kg_by_model, fig_lift_vs_size, fig_mrr_vs_size
from biokgsuite.io import Outputs, find_config, load_config, require
from biokgsuite.viz.style import setup_style

setup_style()

In [2]:
# Config, parameters and the results writer.
config = load_config(find_config(_root), trial_subset=False)   # graphs are named below, not by the panel trial file
params = config['analysis_params']
G      = require(params, 'grounding')
BASE   = config['_base_dir']

# Every figure, table and checkpoint goes through `out`, as in notebooks 01-07.
out = Outputs('G04_scaling', BASE)

KGS       = require(G, 'kgs')
MODELS    = require(G, 'models')
LABELS    = require(G, 'model_labels') | {'none': 'No KG', 'primekg': 'PrimeKG', 'drkg': 'DRKG',
                                           'biokg': 'BioKG', 'openbilink': 'OpenBioLink', 'hetionet': 'Hetionet'}
FAMILIES  = require(G, 'families')
POOL_SIZE = require(G, 'pool_size')
SHUFFLES  = require(G, 'shuffles')
SEEDS     = require(G, 'seeds')
STRICT    = require(G, 'strict')
N_BOOT    = require(G, 'n_bootstrap')
SEED      = require(params, 'random_seed')
RNG       = np.random.default_rng(SEED)
# BKG_MODE=mock reads the rehearsal runs 02 wrote under runs_mock/; real
# (the default) reads the results and refuses any mock row it finds there.
MOCK      = os.environ.get('BKG_MODE', 'real') == 'mock'
RUN_DIR   = Path(BASE) / 'results' / 'tables' / 'G02_run' / ('runs_mock' if MOCK else 'runs')

PARAMS_B = require(G, 'capacity_models')

### Loading

In [3]:
bench = out.load_checkpoint('G01_benchmark')['benchmark']
runs = validate_runs(load_runs(RUN_DIR, families=['main'], models=list(PARAMS_B), allow_mock=MOCK))
present = {m: b for m, b in PARAMS_B.items() if m in set(runs.model)}
print('rungs run so far:', {LABELS.get(m, m): b for m, b in present.items()})
GROUNDED = {kg: set(bench.pair_id[bench[f'grounded_{kg}']]) for kg in KGS}   # the graph has evidence for the positive

rungs run so far: {'Llama-3.2-1B': 1, 'Llama-3.2-3B': 3, 'Llama-3.1-8B': 8, 'Llama-3.3-70B': 70}


## 1. Capacity series

In [4]:
cap = capacity_table(runs, present, ['none'] + KGS, N_BOOT, RNG, LABELS)
display(cap.pivot_table(index=['params_b', 'model_label'], columns='arm_label', values='mrr').round(3))
out.table(cap, 'capacity')

,arm_label,BioKG,DRKG,No KG,PrimeKG
params_b,model_label,,,,
1.0,Llama-3.2-1B,0.312,0.322,0.306,0.316
3.0,Llama-3.2-3B,0.371,0.386,0.338,0.378
8.0,Llama-3.1-8B,0.483,0.507,0.340,0.543
70.0,Llama-3.3-70B,0.556,0.564,0.335,0.641


PosixPath('/home/moline/biokgsuite-bib/results/tables/G04_scaling/capacity.csv')

## 2. Pooled versus grounded lift


In [5]:
scaling = scaling_table(runs, present, GROUNDED, N_BOOT, RNG)
scaling['model_label'] = scaling.model.map(LABELS).fillna(scaling.model)
display(scaling.set_index('model_label')[['params_b', 'mrr_none', 'mrr_kg', 'mrr_kg_grounded', 'lift_pooled', 'lift_grounded', 'n_pairs', 'n_pairs_grounded']].round(3))
out.table(scaling, 'scaling')


,params_b,mrr_none,mrr_kg,mrr_kg_grounded,lift_pooled,lift_grounded,n_pairs,n_pairs_grounded
model_label,,,,,,,,
Llama-3.2-1B,1.0,0.306,0.317,0.320,0.011,0.019,128,88
Llama-3.2-3B,3.0,0.338,0.378,0.409,0.041,0.066,128,88
Llama-3.1-8B,8.0,0.340,0.511,0.623,0.171,0.285,128,88
Llama-3.3-70B,70.0,0.335,0.587,0.747,0.252,0.416,128,88


PosixPath('/home/moline/biokgsuite-bib/results/tables/G04_scaling/scaling.csv')

### Visualisation

In [6]:
out.figure(fig_capacity(cap, chance_mrr(POOL_SIZE), KGS), 'G04a_capacity');
out.figure(fig_lift_vs_size(scaling), 'G04b_lift_vs_size');
out.figure(fig_mrr_vs_size(scaling), 'G04c_mrr_vs_size');
out.figure(fig_kg_by_model(runs, present, KGS, LABELS), 'G04d_kg_by_rung');

  saved G04a_capacity.pdf and .png (600 dpi)


  saved G04b_lift_vs_size.pdf and .png (600 dpi)


  saved G04c_mrr_vs_size.pdf and .png (600 dpi)


  saved G04d_kg_by_rung.pdf and .png (600 dpi)


### Checkpoint

In [7]:
out.checkpoint({'capacity': cap, 'scaling': scaling, 'rungs': present})

PosixPath('/home/moline/biokgsuite-bib/results/checkpoints/G04_scaling.pkl')

In [8]:
# Memory cleanup: free state before the next notebook in the pipeline.
for _v in ('kgs', 'graphs', 'dossiers', 'runs', 'bench'):
    globals().pop(_v, None)
gc.collect()
print('done:', out.notebook)

done: G04_scaling
